# Welcome to Evidence Graph

**Evidence Graph** is a feature of mercury-graph for building traceable knowledge graphs from text, combining an extraction model
([GLiNER2](https://github.com/fastino-ai/GLiNER2)) with LLM agents.

**Why traceable?** When an LLM answers a question, you can believe the answer or not, but you cannot check it. Here, every answer can
be audited: you can ask the agent what it did, see which tools it called, with which arguments and what they returned, and then look
at the graph and the documents yourself. Every relationship found points back to the text it comes from.

This notebook takes the point of view of a **library user**: you create an *Endpoint*, start it and use it **from the outside**, through
its HTTP API, exactly as any other client (or agent) would. You do not need to know HTTP: the `RemoteEndpoint` client
included in mercury-graph does it for you. You will:

1. create the *welcome* Endpoint from the template shipped with the `mge` command line tool,
2. understand its ontology,
3. pilot it and serve it over HTTP,
4. explore its capabilities and navigate its documents and ontologies,
5. extract entities and relationships from its documents,
6. talk to an LLM agent that uses all of the above as tools.

> **Developer preview.** Evidence Graph is under active development: APIs and file formats may change. Section 11 lists what does not
> work yet.

## 0. Configuration

Everything that depends on your machine is in the next cell. The Endpoint folder is created **in the current directory**, and `PORT` is
the HTTP port used from section 5 on.

In [ ]:
import json, os, sys

import pandas as pd

ENDPOINT_PATH = 'welcome'       # Endpoint folder. Its name can only contain letters, numbers and underscores.
PORT          = 8765            # Port where `mge serve` will listen.
URL           = 'http://127.0.0.1:%d' % PORT

# The `mge` command is installed next to the Python interpreter that runs this notebook.
MGE = os.path.join(os.path.dirname(sys.executable), 'mge')

# Use litellm's local model cost map instead of downloading it from GitHub.
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'

## 1. The main ideas

An **Endpoint** is a folder that declares a set of pieces, connects them to each other and serves them over HTTP:

| Piece | What it does | In the welcome template |
|---|---|---|
| **Source** | A set of documents, exposed as a tree (files, sections, paragraphs) that is navigated one level at a time. | `md_example`, `mediawiki_example`, `pdf_example` |
| **Ontology** | A graph of concepts: entity types, relationship types, known instances. | `entities`, `relationships`, `known_ids` |
| **Formalizer** | Proposes the entities and relationships found in a text, using the ontologies as labels. | `GLiNER2` |
| **Evidence graph** | Aggregates what is found in the sources into a graph. | `e_graph` *(not functional yet)* |
| **Agent** | An LLM that can call the other pieces as tools. | `reader`, `gru` |

A **graph** is a set of nodes connected by edges: entities (a person, a school) and the relationships between them (*teaches at*).
Evidence Graph uses graphs for two things. The ontologies describe what can be found: the types of entities and relationships, and the
instances already known. The Evidence graph will hold what has been found in the documents, with each node and edge pointing to the
text it comes from.

**For you, everything is a capability.** Each piece publishes *capabilities*: functions with a unique name, a description and some
arguments, such as `children_by_idx_md_example` or `chat_with_reader`. You run any of them in the same way, with `run()`, whatever
piece is behind it: a Source, an ontology, the Formalizer or an agent. Agents use the very same capabilities as their tools.

Since the Endpoint is served over HTTP, it can be used from any language or application, not only from Python. In Python, the
`RemoteEndpoint` client makes it look like any other library.

## 2. Requirements

**mercury-graph from the repository.** Evidence Graph is not published on PyPI yet, so install it from the `develop` branch:

```bash
git clone https://github.com/BBVA/mercury-graph.git
cd mercury-graph
git checkout develop
python -m venv .venv
source .venv/bin/activate
pip install -e .
```

**JupyterLab** in the same environment: `pip install jupyterlab`.

**An LLM for the agents** (only for section 10). The simplest free option is [Ollama](https://ollama.com):

```bash
brew install ollama
brew services start ollama      # or run `ollama serve` in a terminal if you cannot register services
ollama pull qwen3:8b
```

The first time the Endpoint is piloted (section 5), the GLiNER2 model is downloaded from Hugging Face. This can take a few minutes.

The next cell checks your environment.

In [ ]:
import mercury.graph as mg

print('mercury-graph version:', mg.__version__)
print('mge found:', os.path.isfile(MGE))

`ollama list` shows the models available to the agents. Sections 3 to 9 work without Ollama, section 10 needs it running with
`qwen3:8b`.

In [ ]:
!ollama list

## 3. Create the Endpoint

Installing mercury-graph also installs **`mge`**, the command line tool that creates, pilots and serves Endpoints. `mge --help` lists
its commands, and the [`mge` reference](https://bbva.github.io/mercury-graph/site/reference/evidence_cli/) explains them. This notebook
uses `new`, `pilot`, `summary` and `serve`.

To create an Endpoint, go to the folder where you want it and run `mge new` with its name:

```bash
mge new welcome
```

This copies the Endpoint template into a new folder, `welcome`. The template is a small, complete example: a fictional school
(Riverside High School, its robotics club and a science fair) described in three document formats. The next cell does the same from
the notebook.

To start from scratch again, delete the folder and run the cell again.

In [ ]:
if os.path.exists(ENDPOINT_PATH):
    print('The folder "%s" already exists, skipping `mge new`.' % ENDPOINT_PATH)
else:
    !{MGE} new {ENDPOINT_PATH}

The folder contains:

- **`mge_endpoint.jsonc`**, the main configuration file. Each section (`sources`, `ontologies`, `formalizers`, `evidence_graphs`,
  `agents`, `custom_agentics`) lives in its own `.jsonc` file, included with `{"$ref": "./sources.jsonc"}`.
- **`markdown/`, `xml/`, `pdf/`**: the same school documents as Markdown files, as a MediaWiki XML export and as PDF files.
- **`ontology_source/`**: the tab-separated files used to initialize the ontologies.
- **`mge_endpoint.free`**: the lock. It becomes `mge_endpoint.locked` while a process pilots or serves the Endpoint.

`.jsonc` is JSON with comments, with one restriction: **comments must take a whole line** starting with `//`. A comment after a value on
the same line breaks the file.

In [ ]:
for name in sorted(os.listdir(ENDPOINT_PATH)):
    print(name)

This is the main file without its comments. `expose` lists the pieces whose capabilities are published over HTTP.

In [ ]:
with open(os.path.join(ENDPOINT_PATH, 'mge_endpoint.jsonc')) as f:
    print(''.join(line for line in f if not line.strip().startswith('//')))

### The Sources

A **Source** turns a set of documents into a tree that can be navigated (section 7). Its `type` says how it gets there:

- `markdown_tree` uses the Markdown files in `dst_path` as they are. It has no `src_path`.
- `xml_stream` converts a MediaWiki XML export (`src_path`) into Markdown files in `dst_path` during the pilot.
- `pdf_mirror` mirrors a folder of PDF files (`src_path`) as Markdown files in `dst_path`, converting each PDF the first time it is
  accessed.

The Sources are declared in `sources.jsonc`. These are the main fields of the three in the template:

In [ ]:
with open(os.path.join(ENDPOINT_PATH, 'sources.jsonc')) as f:
    sources = json.loads(''.join(line for line in f if not line.strip().startswith('//')))

pd.DataFrame([{'source': name, **{k: conf['extra_args'].get(k) or '-' for k in ['type', 'src_path', 'dst_path', 'extensions']}}
              for name, conf in sources.items()])

**To use your own documents**, edit `sources.jsonc` and point `src_path` (or `dst_path` for `markdown_tree`) to them:

- Relative paths are relative to the Endpoint folder. Real corpora are usually large and live outside it, so use absolute paths.
- A `dst_path` that does not exist is created.
- `extensions` filters which files are indexed.

The comments in `sources.jsonc` explain every field, including the optional ones (a custom PDF converter, the clustering of large
corpora, a ChromaDB vector store). After changing the configuration, pilot the Endpoint again (section 11).

## 4. Understand the ontology

The Formalizer can only find what the ontologies define. The template ships an ontology for its example documents: the people,
organizations, events, places and projects of a school. There are three ontologies, initialized from the tab-separated files in
`ontology_source/`:

- **`entities`**: entity types, each one with a `definition`.
- **`relationships`**: relationship types, each one with its source type (`src`), destination type (`dst`) and a `definition`.
- **`known_ids`**: concrete, known instances and the edges between them.

In [ ]:
def read_ontology_file(name):
    """ Reads one of the tab-separated files that initialize the ontologies. """

    return pd.read_csv(os.path.join(ENDPOINT_PATH, 'ontology_source', name), sep = '\t')


entities = read_ontology_file('entities_concepts.csv')
entities

In [ ]:
relationships = read_ontology_file('relationships_concepts.csv')
relationships

Identifiers use `|` to build a hierarchy: `person|teacher` is a kind of `person`, and `place|room|laboratory` a kind of `place|room`.
Relationship types can have subtypes too: `works_on|leads`. The hierarchy organizes the ontology for navigation. The Formalizer, however,
sees every identifier as an independent label.

`known_ids` holds instances. Their identifiers start with their type (`person|teacher|Elena Ruiz`), and the edges between them use the
relationship types.

In [ ]:
print(read_ontology_file('known_ids_nodes.csv'))
print()
print(read_ontology_file('known_ids_edges.csv'))

**To change an ontology**, edit its files and keep these rules:

- the files are **tab-separated**,
- parents are written before their children (`person` before `person|teacher`),
- the files are only read when the ontology has not been saved yet. The first pilot saves it in `graphs/<ontology>.pickle`, and from then
  on the saved version wins. To apply new files, stop the server, delete the pickle and pilot again.

## 5. Pilot and serve the Endpoint

**Piloting** takes the Endpoint to a target state. `ALL_READY` (state 100) means that all pieces are loaded, connected to each other and
ready: sources indexed, ontologies loaded and the GLiNER2 model in memory.

`ALL_READY` checks the configuration, not the external services: it does not contact Ollama and it does not convert every document in
advance.

In [ ]:
!{MGE} pilot {ENDPOINT_PATH} ALL_READY

`mge summary` shows every piece of the Endpoint and its state.

In [ ]:
!{MGE} summary {ENDPOINT_PATH}

**Serving** publishes the Endpoint over HTTP. `mge serve` keeps running until you stop it, so it cannot run inside a notebook cell.
Run the command printed below **in a terminal**, using the same Python environment, and keep it open.

While it serves, the Endpoint is locked for other processes. To stop it, press `Ctrl+C` and wait for `Finished server process`: the lock
is released at the very end.

Note that `mge serve` listens on all network interfaces, so the Endpoint is reachable from your network while it runs.

In [ ]:
print('export LITELLM_LOCAL_MODEL_COST_MAP=True')
print('%s serve %s ALL_READY %d' % (MGE, os.path.abspath(ENDPOINT_PATH), PORT))

Once it is running, connect to it with `RemoteEndpoint`. It only needs the URL. If the server is still starting, you get an error
saying that no Endpoint is answering: wait a few seconds and run the cell again.

In [ ]:
from mercury.graph.evidence.remote import RemoteEndpoint

ep = RemoteEndpoint(URL)

## 6. The capabilities

The Endpoint publishes the **capabilities** of all its exposed pieces. Their names are unique and follow the pattern
`<function>_<piece name>`.

In [ ]:
ep.list_capabilities()

`describe()` shows what a capability does and which arguments it takes:

In [ ]:
ep.describe('hint_nodes_GLiNER2')

Each capability is described in the **tool format of the OpenAI API**, the same format that [litellm](https://docs.litellm.ai) accepts
to give tools to an LLM. That is why any piece, even another agent, can be used as a tool by an agent: its capabilities are already tool
definitions. The design favors narrow agents that do one thing well and delegate the rest. `ep.capabilities` keeps them in that
format:

The Endpoint also serves interactive API documentation at `/docs` (`http://127.0.0.1:8765/docs` with the default port).

Two capabilities are not functional yet: `crawl_e_graph` raises an error, and `is_a_GLiNER2` always returns 0.

In [ ]:
print(json.dumps(ep.capabilities[0], indent = 2))

## 7. Navigate a Source

`run()` runs one capability: give it the name of the capability and its argument. When a capability takes a single text, pass the
text directly. Otherwise, pass a dictionary with its arguments.

A Source is navigated like a file explorer, **one level at a time**, with two capabilities:

- `children_by_idx_<source>`: the indices below an index ("list this folder"),
- `object_by_idx_<source>`: what is at an index ("open this file").

An index is a path joined with `|`, and it **always starts with the name of the Source**. This is how an agent can explore a corpus much
larger than its context window.

In [ ]:
ep.run('children_by_idx_md_example', 'md_example')

Inside a Markdown file, the tree follows its headings: `header_<level>_<n>`. A section contains its paragraphs and its subsections, in
order.

In [ ]:
ep.run('children_by_idx_md_example', 'md_example|school.md')

In [ ]:
ep.run('children_by_idx_md_example', 'md_example|school.md|header_1_1')

Opening a section returns its title. Paragraphs are containers too: each one has a single leaf, `text_<n>`, with its whole text. Leaves
are the only nodes that have a `content`.

In [ ]:
print(ep.run('object_by_idx_md_example', 'md_example|school.md|header_1_1|header_2_1'))
print(ep.run('object_by_idx_md_example', 'md_example|school.md|header_1_1|header_2_1|paragraph_4'))
print(ep.run('children_by_idx_md_example', 'md_example|school.md|header_1_1|header_2_1|paragraph_4'))

This paragraph is written over two lines in `school.md`, but its text leaf returns it whole.

In [ ]:
ep.run('object_by_idx_md_example', 'md_example|school.md|header_1_1|header_2_1|paragraph_4|text_6')

Links and images are leaves of their own, next to the text of their paragraph. They keep the exact link, so a program (or an agent) knows
what links where.

In [ ]:
paragraph = 'md_example|robotics_club.md|header_1_1|paragraph_1'

for leaf in ep.run('children_by_idx_md_example', paragraph):
    print(leaf.split('|')[-1], '->', ep.run('object_by_idx_md_example', leaf)['content'])

Mistakes do not raise errors: an index that does not exist returns `None`. A common one is forgetting the name of the Source.

In [ ]:
print(ep.run('children_by_idx_md_example', 'school.md'))

With the same two capabilities, a program can walk a whole file. This function collects the text leaves below an index: section titles
and whole paragraphs. It skips link and image leaves, whose text is already part of their paragraph.

In [ ]:
def collect_texts(source, index):
    """ Walks the tree of `source` below `index` and returns the content of its text leaves, in order. """

    if index.split('|')[-1].startswith('text_'):
        return [ep.run('object_by_idx_%s' % source, index)['content']]

    texts = []
    for child in ep.run('children_by_idx_%s' % source, index) or []:
        texts.extend(collect_texts(source, child))

    return texts


school_texts = collect_texts('md_example', 'md_example|school.md')
for text in school_texts:
    print('-', text)

**The other two Sources** hold the same documents in other formats. Both are converted to Markdown and then navigated exactly like
`md_example`:

- `mediawiki_example` (`xml_stream`) converts a MediaWiki XML export during the pilot. Links between wiki pages become plain text, so
  their names are kept but not their targets.
- `pdf_example` (`pdf_mirror`) converts each PDF the first time it is accessed. The default converter needs `pip install pdf-oxide`, which
  is not installed with mercury-graph. The reading order of the converted text may be wrong, which breaks the heading hierarchy.

For now, PDF is the least reliable format.

## 8. Navigate the ontologies

Ontologies are navigated in the same way: `children_by_idx_<ontology>` for the hierarchy and `node_by_idx_<ontology>` for the properties
of a node. Indices start with the name of the ontology.

In [ ]:
print(ep.run('children_by_idx_entities', 'entities'))
print(ep.run('children_by_idx_entities', 'entities|place'))
print(ep.run('children_by_idx_entities', 'entities|place|room'))
print(ep.run('node_by_idx_entities', 'entities|place|room|laboratory'))

In [ ]:
print(ep.run('children_by_idx_relationships', 'relationships|works_on'))
print(ep.run('node_by_idx_relationships', 'relationships|works_on|leads'))

In `known_ids`, instances are navigated by type. Edges live under the special branch `_edge_`, and the index of an edge is
`known_ids|_edge_|<source>||<destination>||<edge id>`.

In [ ]:
print(ep.run('children_by_idx_known_ids', 'known_ids|person|student'))
print(ep.run('node_by_idx_known_ids', 'known_ids|_edge_|person|student|Noah Kim||project|Helios Solar Car||id_noah_leads_helios'))

## 9. Extract entities and relationships

The Formalizer proposes the entities (`hint_nodes_GLiNER2`) and relationships (`hint_edges_GLiNER2`) that it finds in a text, using the
ontologies as labels. The answer includes every label, so let's print only the ones with results.

In [ ]:
sentence = ('Elena Ruiz teaches science at Riverside High School. '
            'Noah Kim and Priya Shah study at Riverside High School.')

for label, found in ep.run('hint_nodes_GLiNER2', sentence).items():
    if found:
        print('%-22s %s' % (label, found))

Relationships are grouped by the **definition** of the relationship type. Each candidate has its two ends, and some candidates come with
one end empty. This helper keeps the complete ones and maps each definition back to its identifier.

In [ ]:
relationship_ids = dict(zip(relationships['definition'], relationships['id']))


def complete_edges(text):
    """ Returns the relationships proposed for `text` that have both ends, as (source, relationship, destination) tuples. """

    edges = []
    for definition, candidates in ep.run('hint_edges_GLiNER2', text).items():
        for candidate in candidates:
            (_, src), (_, dst) = candidate.items()
            for s in src:
                for d in dst:
                    edges.append((s, relationship_ids.get(definition, definition), d))

    return edges


complete_edges(sentence)

Until the Evidence graph can crawl the Sources by itself, you can do it by hand: take the texts collected in section 7 and run the
Formalizer on each of them.

In [ ]:
nodes = []
edges = []

for text in school_texts:
    for label, found in ep.run('hint_nodes_GLiNER2', text).items():
        nodes.extend((entity, label, text) for entity in found)

    edges.extend(edge + (text,) for edge in complete_edges(text))

nodes = pd.DataFrame(nodes, columns = ['entity', 'type', 'text'])
nodes.groupby(['entity', 'type']).size().rename('mentions').reset_index()

In [ ]:
edges = pd.DataFrame(edges, columns = ['src', 'relationship', 'dst', 'text']).drop_duplicates()
edges

Read these tables as **candidates**, not facts. With an ontology that fits the documents, most of them are right, but typical mistakes are:

- relationship types proposed even when the text does not state them,
- the wrong subtype for a person (a student labeled as a teacher),
- the same relationship proposed more than once.

Keeping the `text` column is what makes the result traceable: every candidate points to the text it comes from, so it can be checked.

## 10. Talk to an agent

An agent exposes a single capability (`chat_with_<agent>`) with one argument, `content`. Its configuration (in `agents.jsonc`) sets the
model, the instructions (`you_are`, `you_must`) and the pieces it can use as tools. The `reader` agent can use `mediawiki_example`,
`GLiNER2` and `e_graph`, and its instructions tell it to use the Formalizer to find what the user asks about.

This section needs Ollama running with the `qwen3:8b` model (see section 2).

In [ ]:
question = 'Which people and organizations are mentioned in this sentence: "Elena Ruiz teaches science at Riverside High School."?'

ep.run('chat_with_reader', question)

The agent does not call the tools itself. When the LLM asks for a tool, the **Endpoint** runs it, appends the result to the conversation and
calls the agent again, until the LLM gives a final answer. With `tool_call_history` enabled (the default), the response includes that
conversation. `run()` returns only the final answer, and keeps the complete response in `ep.last_response`.

In [ ]:
for i, message in enumerate(ep.last_response.get('history', [])):
    for tool_call in message.get('tool_calls') or []:
        print('[%d] %-9s asks for %s %s' % (i, message['role'], tool_call['function']['name'], tool_call['function']['arguments']))

    if message['role'] != 'assistant':
        print('[%d] %-9s %s' % (i, message['role'], str(message.get('content'))[:200]))

Agents trust their tools completely: they repeat what a tool returns, even when it is wrong. That is why the trace matters: check
what the agent called and what it got back before trusting its answer.

This question works because it matches what the `reader` is set up to do. Open questions about the documents, such as *Who teaches
at Riverside High School?*, **are not answered reliably yet**. LLM answers vary from run to run, and this is what we have observed
with the template configuration:

- The `reader` instructions in the template are placeholders. The agent runs the Formalizer on the *question* instead of reading
  the documents, and then answers that it has no information, or even makes an answer up.
- It may choose `crawl_e_graph`, which is not implemented yet. A failing tool ends the whole request with an error, and `ep.last_response` keeps the history up to
  the failure.
- `max_tool_calls_per_query` (10 in `mge_endpoint.jsonc`) counts both tool calls and LLM calls, which limits how deep an agent can
  navigate a Source.

To go further, edit `you_are`, `you_must` and `tools` in `agents.jsonc` (for example, to use `md_example`). After any configuration
change, stop the server, pilot again and serve again.

Try your own questions: change `question` above and run the cells again.

## 11. Current limitations and next steps

What does not work yet:

- The Evidence graph (`e_graph`) cannot crawl the Sources yet (`crawl_e_graph`), so the graph is not built automatically.
- Extraction quality: the Formalizer proposes candidates that need to be checked (section 9).
- Agents run their tools correctly, but their answers are not reliable yet (section 10).
- PDF conversion needs an extra package and may break the reading order (section 7).

To use your own documents:

1. Point a Source to your files in `sources.jsonc` (`src_path`, `dst_path`, `type`). Relative paths are relative to the Endpoint folder.
2. Write an ontology for your domain, following the rules in section 4.
3. Give your agents instructions and tools that match your documents in `agents.jsonc`.
4. Stop the server, pilot and serve again.

When you finish, stop `mge serve` with `Ctrl+C` in its terminal. If a process is interrupted before releasing the lock, `mge unlock`
releases it.